# Client Mode vs Cluster Mode: Deep Comparison and Use Cases

**Module**: 12_advanced_topics  
**Notebook**: 03_cluster_vs_client_mode.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: Complete comparison of Client Mode vs Cluster Mode across Standalone, YARN, and Kubernetes.
- **Why does this topic exist?**: Choosing the wrong deploy mode can crash interactive terminals or isolate scheduled batch jobs.
- **What problem does it solve?**: Understanding where the Driver runs and why network latency matters.
- **Where is it used in real-world Data Engineering?**: Configuring Airflow SparkSubmitOperator in enterprise environments.

---

## 2. Client vs Cluster Mode Architecture
```
CLIENT MODE:
[Submitting Machine (Jupyter / Laptop)]  <--- High Network Latency Risk!
  - Runs DRIVER Process
  - Requires persistent low-latency connection to cluster!
       | RPC
       v
[Remote Cluster (Workers / Executors)]

CLUSTER MODE:
[Submitting Machine] ---> Submits job manifest to Master and DISCONNECTS (Fire & Forget!)
[Remote Cluster]
  - Node 1: Worker launches DRIVER JVM inside cluster!
  - Node 2, 3: Workers launch EXECUTORS!
```

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Client_Cluster_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
print("Our Docker JupyterLab environment runs in: CLIENT MODE")
print("Driver Host:   ", spark.conf.get("spark.driver.host"))
print("Master URL:    ", spark.conf.get("spark.master"))

---

## Experiment: Deploy Mode Decision Guide

- **1. What are we testing?**: Print decision rules for Client vs Cluster mode.
- **2. What do we expect?**: Client for interactive/dev; Cluster for production.

In [ ]:
print("""
Rule 1: If using JupyterLab, spark-shell, or PySpark REPL -> MUST USE CLIENT MODE.
Rule 2: If using Airflow, cron, or production scheduled ETL -> ALWAYS USE CLUSTER MODE.
""")

**3. What happened?**: Explained architectural deployment rules.  
**4. Why did it happen?**: Client mode binds Driver to submitting machine; Cluster mode decouples submission from execution.  
**5. Practical Engineering Lesson**: Never use client mode for production scheduled jobs over remote VPNs or CI/CD pipelines.

---

## Practical Exercises

### Exercise 1 (Beginner): Identify Lab Mode
**Problem Statement**: What deploy mode is active in this notebook?  
**Input / Context**: Mode

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Cluster Mode Log Access
**Problem Statement**: How do you view driver logs in cluster mode?  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): Network Partition Impact
**Problem Statement**: What happens if submitting machine disconnects in client mode?  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Identify Lab Mode):
```python
print('Client mode (Jupyter hosts Driver; workers host Executors).')
```

#### Solution 2 (Cluster Mode Log Access):
```python
print('Via yarn logs -applicationId <id> or Spark History Server (18080).')
```

#### Solution 3 (Network Partition Impact):
```python
print('Driver process terminates; entire cluster application is aborted immediately.')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **Client Mode**: Driver on client; required for notebooks.
- **Cluster Mode**: Driver on worker; fire-and-forget for production.

### Essential Technical Terms
- `Deploy Mode`: Client vs Cluster flag.
- `Fire-and-Forget`: Submitting job and disconnecting.

### Common Engineering Mistakes
- **Running client mode over high-latency VPN**: Heartbeat timeouts kill job.

### Interview Questions on This Topic
- **Q: What is the fundamental difference between Client mode and Cluster mode?**
  - *A*: In Client mode, the Driver runs on the submitting machine; in Cluster mode, the Driver runs on a worker node inside the cluster.

### What to Study Next
Open 04_resource_allocation.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')